### BERT Classifier

##### Imports

In [ ]:
import os
import json
import torch
import evaluate
import numpy as np
import random

from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoConfig, AutoModel, TrainingArguments, TrainerCallback, Trainer
from collections import Counter
from torch import nn
from torch.utils.data import DataLoader
from safetensors.torch import save_file, load_file
from pathlib import Path

##### Model Training

In [ ]:
MODEL_NAME = "deepset/gbert-base"   # or "bert-base-german-cased" / "xlm-roberta-base"
SAVE_DIR = "combined_gbert_base_model"
BEST_MODEL_DIR = SAVE_DIR+"/best_model"
BEST_MODEL_METRIC_PATH = BEST_MODEL_DIR+"/best_metric.json"
DATASET_DIR = "../ner_data/"
N_EPOCHS = 5
EXTEND_EPOCHS = 0
MAX_LEN = 64
LR = 3e-5
SEED = 42

In [ ]:
# Split dataset into train/validation/test
# --- File paths ---
combined_data_path = DATASET_DIR + "combined_data.jsonl"
train_path = DATASET_DIR + "train.jsonl"
val_path = DATASET_DIR + "val.jsonl"
test_path = DATASET_DIR + "test.jsonl"

# --- Split ratios ---
train_ratio = 0.8
val_ratio = 0.15
test_ratio = 0.05

# --- Read all lines ---
data = []
with open(combined_data_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():   # skip blank lines
            continue
        data.append(json.loads(line))

print("Data size:", len(data))
# --- Shuffle for randomness ---
random.seed(SEED)
random.shuffle(data)

# --- Compute split sizes ---
n = len(data)
train_end = int(train_ratio * n)
val_end = train_end + int(val_ratio * n)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

# --- Write splits back to disk ---
def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

write_jsonl(train_path, train_data)
write_jsonl(val_path, val_data)
write_jsonl(test_path, test_data)

In [ ]:
# Build label list from the dataset
labels = set()
with open(combined_data_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():   # skip blank lines
            continue
        record = json.loads(line)
        labels.update(record.get("tags", []))

if "O" in labels:
    labels.remove("O")

# keep deterministic ordering, "O" at the beginning
labels = ["O"] + sorted(labels)
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(labels)
print("Total labels:", num_labels)
print(labels)

In [ ]:
# Load datasets
data_files = {
    "train": DATASET_DIR + "train.jsonl",
    "validation": DATASET_DIR + "val.jsonl",
    "test": DATASET_DIR + "test.jsonl",
}
raw = load_dataset("json", data_files=data_files)

In [ ]:
# Tokenizer & alignment
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )
    all_labels = []
    for i, tags in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        aligned = []
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)           # special tokens
            elif wid != prev:
                aligned.append(label2id[tags[wid]])
            else:
                aligned.append(-100)           # subword continuation
            prev = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized = raw.map(tokenize_and_align, batched=True, remove_columns=["tokens","tags"])

In [ ]:
# Model: BERT + linear
class BertTagger(nn.Module):
    def __init__(self, name: str, num_labels: int, dropout: float = 0.15):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq = self.dropout(outputs.last_hidden_state)      # [B, T, H]
        logits = self.classifier(seq)                      # [B, T, L]

        loss = None
        if labels is not None:
            # Flatten to compute token-level CE, ignoring -100 positions
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            return {"loss": loss, "logits": logits}
        else:
            return logits

model = BertTagger(name=MODEL_NAME, num_labels=num_labels, dropout=0.2)

In [ ]:
# Callback to save best model based on eval metric
class SaveBestModelCallback(TrainerCallback):
    def __init__(self, metric_name="eval_f1", greater_is_better=True, best_metric = None):
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = best_metric

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        # print("Entering callback!", metrics)
        if metrics is None:
            print("Metric is None!")
            return
        elif self.metric_name not in metrics:
            print("Metric_name is None!")
            return

        current_metric = metrics[self.metric_name]
        
        # Check if metric improved
        improved = (
            self.best_metric is None or
            (self.greater_is_better and current_metric > self.best_metric) or
            (not self.greater_is_better and current_metric < self.best_metric)
        )
        # print("Didn't improve!")

        if improved:
            print(f"New best {self.metric_name}: {current_metric:.4f} → saving model")
            self.best_metric = current_metric
            
            # Save model and tokenizer
            model = kwargs.get("model")
            tokenizer = kwargs.get("tokenizer")
            
            # Save best metric
            with open(BEST_MODEL_METRIC_PATH, "w", encoding="utf-8") as f:
                json.dump(
                    {
                        "best_metric": self.best_metric,
                        "metric_name": self.metric_name,
                        "greater_is_better": self.greater_is_better,
                        "global_step": getattr(state, "global_step", None),
                        "epoch": float(getattr(state, "epoch", 0.0)) if getattr(state, "epoch", None) is not None else None,
                    },
                    f, indent=2,
                )

            # Save model weights
            save_file(model.state_dict(), os.path.join(BEST_MODEL_DIR, "model.safetensors"))
            # Save tokenizer
            if tokenizer is not None:
                tokenizer.save_pretrained(BEST_MODEL_DIR)
            else:
                print("Tokenizer is none!")

In [ ]:
# Create directory to store best metric (if it doesn't exist)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
best_metric = None
if os.path.exists(BEST_MODEL_METRIC_PATH): # Check if the file exists
    with open(BEST_MODEL_METRIC_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
        best_metric = data.get("best_metric", None) # Safely get best_metric from JSON
        
    print(f"Loaded previous best metric: {best_metric:.4f}")
else:
    print("No best metric file found. Starting fresh.")

callback = SaveBestModelCallback(metric_name="eval_f1", greater_is_better=True, best_metric=best_metric)

In [ ]:
# Metrics with seqeval
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, (list, tuple)):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)

    # mask out special/pad/subword positions
    mask = labels != -100
    true_preds = [p[m].tolist() for p, m in zip(preds, mask)]
    true_labels = [l[m].tolist() for l, m in zip(labels, mask)]

    # map ids -> tag strings before feeding to seqeval
    true_preds_str  = [[id2label[i] for i in row] for row in true_preds]
    true_labels_str = [[id2label[i] for i in row] for row in true_labels]

    from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score
    return {
        "precision": precision_score(true_labels_str, true_preds_str),
        "recall":    recall_score(true_labels_str, true_preds_str),
        "f1":        f1_score(true_labels_str, true_preds_str),
        "accuracy":  accuracy_score(true_labels_str, true_preds_str),
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    eval_strategy="epoch",          
    save_strategy="epoch",          
    load_best_model_at_end=False,    
    # metric_for_best_model="f1",    
    # greater_is_better=True,         
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=LR,
    num_train_epochs=N_EPOCHS,
    weight_decay=0.01,
    fp16=True,
    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

# Huggingface Trainer
trainer = Trainer(
    model=model,                      
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics, 
    callbacks=[callback],
)

In [ ]:
# Start or resume training
checkpoints = list(Path(SAVE_DIR).glob("checkpoint-*"))
checkpoints = sorted(checkpoints, key=lambda x: int(x.name.split("-")[1])) # Sort by the step number
if checkpoints:
    # Extend num_train_epochs beyond the previous total
    trainer.args.num_train_epochs += EXTEND_EPOCHS
    print(f"Found checkpoint {checkpoints[-1]}, resuming training")
    trainer.train(resume_from_checkpoint=True)
else:
    print("No checkpoint found, starting fresh training")
    trainer.train()

# trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label maps
with open(SAVE_DIR+"/labels.json","w",encoding="utf-8") as f:
    json.dump({"labels": labels}, f, ensure_ascii=False, indent=2)

##### Evaluation

In [ ]:
# Load labels
with open(SAVE_DIR+"/labels.json","r",encoding="utf-8") as f:
    labels = json.load(f)["labels"]
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BertTagger(name=MODEL_NAME, num_labels=len(labels))

# Loading best model if exists
best_model_file = os.path.join(BEST_MODEL_DIR, "model.safetensors")
if os.path.exists(best_model_file): # If best model exists
    print("Loading best model!")
    state_dict = load_file(BEST_MODEL_DIR+"/model.safetensors", device=device)
else: # Otherwise load the current one
    print("Loading current model!")
    state_dict = load_file(SAVE_DIR+"/model.safetensors", device=device)

model.load_state_dict(state_dict)
model.to(device).eval()

In [ ]:
# To perform inference on a single title
def classify_title_tags(title, tokenizer, model, id2label, max_len, device):
    words = title.split()
    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    tokenized = tokenized.to(device)

    # Forward pass -> logits -> argmax to get predicted IDs
    model.eval()
    with torch.no_grad():
        out = model(
            input_ids=tokenized["input_ids"],
            attention_mask=tokenized["attention_mask"]
        )
        logits = out["logits"] if isinstance(out, dict) else out     # [1, T, L]
        pred_ids = torch.argmax(logits, dim=-1)                      # [1, T]

    # Align predictions back to words (first subword only)
    word_ids = tokenized.word_ids(batch_index=0)  # length T
    pred_seq = pred_ids[0].detach().cpu().tolist()

    aligned = []
    seen_word = set()
    for i, wid in enumerate(word_ids):
        if wid is None:
            continue  # special tokens [CLS]/[SEP]/[PAD]
        if wid not in seen_word:
            seen_word.add(wid)  # first subword of this word
            tag_id = pred_seq[i]
            tag = id2label[int(tag_id)]
            # wid should be < len(words) when tokenization matches splitting
            if wid < len(words):
                aligned.append((words[wid], tag))

    return aligned

##### Quiz Set

In [ ]:
import pandas as pd
from enum import Enum

In [ ]:
try:
    # Works when running a .py file
    ROOT = Path(__file__).parent.resolve()
except NameError:
    # Works in notebooks / interactive
    ROOT = Path.cwd().resolve()

In [ ]:
class Cols(str, Enum):
    Record_Number = "Record Number"
    Category = "Category"
    Title = "Title"
    Token = "Token"
    Tag = "Tag"
    Token_Tag_List = "Token_Tag_List"

In [ ]:
TEST_DIR = "Test Directory"
df_test = pd.read_csv(
    TEST_DIR,
    sep="\t",
    keep_default_na=False,
    na_values=None, 
    encoding="utf-8",
    header=0,)

In [ ]:
# Perform classification on the quiz data range
start, end = 5000, 30001
df_test = df_test[(df_test[Cols.Record_Number] >= start) & (df_test[Cols.Record_Number] <= end)]

In [ ]:
# To prevent invalid tag assignments for a certain category
allowed_tags_for_category = {}
with open(DATASET_DIR + "allowed_tags_for_category.json","r",encoding="utf-8") as f:
    allowed_tags_for_category = json.load(f)

In [ ]:
mistakes = []
with open("submit_output.tsv", "w", encoding="utf-8") as f:
    columns = ["Record Number", "Category", "Aspect Name", "Aspect Value"]
    f.write("\t".join(columns) + "\n")
    for idx, row in df_test.iterrows():
        # if idx > 5000:
        #     break
        if idx % 1000 == 0:
            print(f"Processing row {idx}")
        record_number = row[Cols.Record_Number]
        category = row[Cols.Category]
        title = "KAT_"+str(category)+" "+row[Cols.Title] # Appending "KAT_1 or KAT_2" to include category information
        tokens = title.split()
        pred_tag = [classify_title_tags(title, tokenizer, model, id2label, MAX_LEN, device)][0]

        # Combine subwords and filter by allowed tags for the category
        final_token_tag = []
        invalid = 0
        ignore_idx = 0
        for token, tag in pred_tag:
            if ignore_idx == 0: # Ignoring the KAT_1 or KAT_2 token
                ignore_idx += 1
                continue
            if tag == "O":
                final_token_tag.append([token, tag])
                continue
            cur_token = token
            cur_tag = tag[2:]
            if category not in allowed_tags_for_category.get(cur_tag, []):
                cur_tag = 'O'
                final_token_tag.append([cur_token, cur_tag])
            elif tag[:2] == 'B-':
                final_token_tag.append([cur_token, cur_tag])
            else:
                if len(final_token_tag) == 0 or final_token_tag[-1][1] != cur_tag:
                    if invalid == 0:
                        mistakes.append([record_number, pred_tag])
                        invalid += 1
                    if len(final_token_tag) == 0:
                        cur_tag = 'O'  # Reset to 'O' if no previous tag to append to
                        final_token_tag.append([cur_token, cur_tag])
                    else:
                        final_token_tag[-1][0] += ' ' + cur_token  # append to previous token
                else:
                    final_token_tag[-1][0] += ' ' + cur_token  # append to previous token
            
        for t in final_token_tag:
            aspect_name = t[1]
            aspect_value = t[0]
            f.write(f"{record_number}\t{category}\t{aspect_name}\t{aspect_value}\n") # Saving in the submission file

print(f"Total mistakes: {len(mistakes)}")

##### Build Enhanced Train Set

In [ ]:
TEST_DIR = "Test Directory"
df_test = pd.read_csv(
    TEST_DIR, 
    sep="\t",
    keep_default_na=False,
    na_values=None, 
    encoding="utf-8",
    header=0,)

print("Total rows:", len(df_test))

In [ ]:
start, end = 5000, 200000-5
df_test = df_test[(df_test[Cols.Record_Number] >= start) & (df_test[Cols.Record_Number] <= end)]

In [ ]:
allowed_tags_for_category = {}
with open(DATASET_DIR + "allowed_tags_for_category.json","r",encoding="utf-8") as f:
    allowed_tags_for_category = json.load(f)

In [ ]:
# To perform pseudo-labeling with strict BIO + confidence checks
@torch.no_grad()
def pseudo_label_title_bio_strict_abort(
    title: str,
    tokenizer,
    model,                
    id2label: dict,
    max_len: int = 128,
    device: str = "cpu",
    tok_thresh: float = 0.90,   # per-token probability threshold
    span_thresh: float = 0.95,  # span-level confidence threshold
    agg: str = "min",           # 'min' or 'mean'
    drop_if_no_span: bool = True
):
    """
    Returns:
      (output_dict_or_None, low_conf_issue, invalid_bio_issue)
    """
    words = title.split()
    if not words:
        return None, False, False

    batch = tokenizer(
        words, is_split_into_words=True,
        truncation=True, padding="max_length", max_length=max_len,
        return_tensors="pt"
    ).to(device)

    model.eval()
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out["logits"] if isinstance(out, dict) else out   # [1, T, L]
    probs  = logits.softmax(dim=-1)[0]                          # [T, L]
    pred_ids = probs.argmax(dim=-1)                             # [T]

    word_ids = batch.word_ids(batch_index=0)
    word_tags, word_confs = [], []
    seen = set()
    for i, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        pid  = int(pred_ids[i].item())
        tag  = id2label[pid]
        conf = float(probs[i, pid].item())
        word_tags.append(tag)
        word_confs.append(conf)

    # strict legality + confidence pre-check
    prev_tag = None
    low_conf_issue = False
    invalid_bio_issue = False

    for tag, conf in zip(word_tags, word_confs):
        if conf < tok_thresh:
            low_conf_issue = True
            return None, low_conf_issue, invalid_bio_issue

        if tag == "O":
            prev_tag = "O"
            continue
        if tag.startswith("B-"):
            prev_tag = tag
            continue
        if tag.startswith("I-"):
            if prev_tag is None or prev_tag == "O" or not (prev_tag.startswith("B-") or prev_tag.startswith("I-")):
                invalid_bio_issue = True
                return None, low_conf_issue, invalid_bio_issue
            if prev_tag.split("-", 1)[1] != tag.split("-", 1)[1]:
                invalid_bio_issue = True
                return None, low_conf_issue, invalid_bio_issue
            prev_tag = tag

    # build spans
    spans = []  # (start, end, type, span_conf)
    i, n = 0, len(word_tags)
    while i < n:
        tag = word_tags[i]
        if tag.startswith("B-"):
            et = tag.split("-", 1)[1]
            j = i + 1
            confs = [word_confs[i]]
            while j < n and word_tags[j] == f"I-{et}":
                confs.append(word_confs[j])
                j += 1
            span_conf = min(confs) if agg == "min" else sum(confs)/len(confs)
            spans.append((i, j-1, et, span_conf))
            i = j
        else:
            i += 1

    # full BIO output
    tags_out = ["O"] * len(words)
    conf_out = [0.0] * len(words)
    kept_any = False
    for s, e, et, sc in spans:
        if sc >= span_thresh:
            tags_out[s] = f"B-{et}"
            conf_out[s] = sc
            for k in range(s+1, e+1):
                tags_out[k] = f"I-{et}"
                conf_out[k] = sc
            kept_any = True

    if not kept_any and drop_if_no_span:
        return None, low_conf_issue, invalid_bio_issue

    return {"tokens": words, "tags": tags_out, "conf": conf_out}, low_conf_issue, invalid_bio_issue

In [ ]:
CONFIDENT_THRESHOLD = 0.9

cnt_added_rows = 0
cnt_low_confidence = 0
cnt_invalid_bio = 0
with open("confident_data.jsonl", "w", encoding="utf-8") as f:
    for idx, row in df_test.iterrows():
        # if idx > 5300:
        #     break
        if idx % 1000 == 0:
            print(f"Processing row {idx}")
        record_number = row[Cols.Record_Number]
        category = row[Cols.Category]
        title = "KAT_"+str(category)+" "+row[Cols.Title] # Appending "KAT_1 or KAT_2" to include category information
        tokens = title.split()
        pred, low_conf_issue, invalid_bio_issue = pseudo_label_title_bio_strict_abort(title, tokenizer, model, id2label, MAX_LEN, device, 
                    tok_thresh=CONFIDENT_THRESHOLD)
        
        cnt_low_confidence += low_conf_issue
        cnt_invalid_bio += invalid_bio_issue
        if pred != None:
            cur_data = {'tokens': pred['tokens'], 'tags': pred['tags']}
            f.write(json.dumps(cur_data, ensure_ascii=False)+"\n")
            cnt_added_rows += 1

In [ ]:
print(cnt_added_rows, cnt_low_confidence, cnt_invalid_bio)